# Zero-Shot Model Comparison - Chronos & Kronos

In [ ]:
import os
from pathlib import Path

REPO_NAME = "ba"
REPO_URL = "https://github.com/bp571/ba.git"

if not Path(REPO_NAME).exists():
    !git clone {REPO_URL}

os.chdir(REPO_NAME)
print(f"Working directory: {os.getcwd()}")

In [ ]:
!pip install -q torch transformers peft gluonts pandas numpy tqdm PyYAML

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Re-Run auf 104-Asset-Universum (energy_assets_filtered.yaml)
SEEDS  = [13, 42, 123, 456, 789]
CONFIG = "config/energy_assets_filtered.yaml"

# Zero-Shot Parameter (HANDOFF: c=80, f=12, s=12)
CONTEXT  = 80
FORECAST = 12

print(f"Seeds:    {SEEDS}")
print(f"Config:   {CONFIG}")
print(f"Context:  {CONTEXT}, Forecast: {FORECAST}")

## Chronos Zero-Shot

In [ ]:
import subprocess, time

for seed in SEEDS:
    print(f"\n{'='*60}\nChronos | Seed {seed}\n{'='*60}")
    t0 = time.time()
    subprocess.run([
        "python", "01_model_comparison/zeroshot/main_chronos.py",
        "--seed", str(seed),
        "--config", CONFIG,
    ], check=True)
    print(f"Seed {seed} done in {time.time()-t0:.1f}s")

## Kronos Zero-Shot

In [ ]:
import subprocess, time

for seed in SEEDS:
    print(f"\n{'='*60}\nKronos | Seed {seed}\n{'='*60}")
    t0 = time.time()
    subprocess.run([
        "python", "01_model_comparison/zeroshot/main_kronos.py",
        "--seed", str(seed),
        "--config", CONFIG,
        "--context", str(CONTEXT),
        "--forecast", str(FORECAST),
    ], check=True)
    print(f"Seed {seed} done in {time.time()-t0:.1f}s")

## Summary

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

def aggregate(model: str, seed: int) -> dict | None:
    p = Path(f"01_model_comparison/results/{model}/seed_{seed}/final_energy_study.json")
    if not p.exists():
        return None
    with open(p) as f:
        data = json.load(f)
    summary = data["summary"]
    maes    = [v["MAE_indicative"]         for v in summary.values() if "MAE_indicative"         in v]
    ics     = [v["IC_TimeSeries_Mean"]     for v in summary.values() if "IC_TimeSeries_Mean"     in v]
    rankics = [v["RankIC_TimeSeries_Mean"] for v in summary.values() if "RankIC_TimeSeries_Mean" in v]
    return {
        "model":    model,
        "seed":     seed,
        "n_assets": data["n_assets_processed"],
        "time_s":   data["processing_time_seconds"],
        "MAE":      np.mean(maes)    if maes    else np.nan,
        "IC":       np.mean(ics)     if ics     else np.nan,
        "RankIC":   np.mean(rankics) if rankics else np.nan,
    }

rows = []
for model in ["chronos", "kronos"]:
    for seed in SEEDS:
        row = aggregate(model, seed)
        if row:
            rows.append(row)

df = pd.DataFrame(rows)
print("\nPer-Seed Ergebnisse")
print("="*70)
print(df.round(4).to_string(index=False))

print("\nAggregat (Mittelwert ± Std über Seeds)")
print("="*70)
agg = df.groupby("model").agg(
    n_assets=("n_assets", "mean"),
    MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"),
    IC_mean=("IC", "mean"),   IC_std=("IC", "std"),
    RankIC_mean=("RankIC", "mean"), RankIC_std=("RankIC", "std"),
    total_time_s=("time_s", "sum"),
).round(4)
print(agg.to_string())

## Kronos Base mit optimalen Parametern (context=120, forecast=6, seed=42)

Fairer Vergleich mit Kronos Fine-tuned — identische Parameter.

In [ ]:
!python 01_model_comparison/zeroshot/main_kronos.py \
    --context 120 \
    --forecast 6 \
    --seed 42 \
    --config config/energy_assets_filtered.yaml

## Vergleich: Kronos Base vs. Kronos Fine-tuned (context=120, forecast=6, seed=42)

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

def load_metrics(path):
    with open(path) as f:
        d = json.load(f)
    summary = d['summary']
    maes    = [v['MAE_indicative']        for v in summary.values() if 'MAE_indicative'        in v]
    ics     = [v['IC_TimeSeries_Mean']    for v in summary.values() if 'IC_TimeSeries_Mean'    in v]
    rankics = [v['RankIC_TimeSeries_Mean'] for v in summary.values() if 'RankIC_TimeSeries_Mean' in v]
    return {
        'MAE':    np.mean(maes),
        'IC':     np.mean(ics),
        'RankIC': np.mean(rankics),
        'RankIC>0': sum(r > 0 for r in rankics),
        'n': len(rankics),
    }

base_path = Path("01_model_comparison/results/kronos/seed_42/final_energy_study.json")
ft_path   = Path("02_finetuning/results/kronos_finetuned/seed_42/final_energy_study.json")

rows = []
if base_path.exists():
    rows.append({'Model': 'Kronos Base', **load_metrics(base_path)})
else:
    print("Kronos Base result not found — run cell above first.")

if ft_path.exists():
    rows.append({'Model': 'Kronos Fine-tuned', **load_metrics(ft_path)})
else:
    print("Kronos Fine-tuned result not found.")

if rows:
    df = pd.DataFrame(rows).set_index('Model')
    df['RankIC>0'] = df.apply(lambda r: f"{int(r['RankIC>0'])}/{int(r['n'])}", axis=1)
    df = df.drop(columns='n')
    df = df.round(4)
    print("\nKronos Base vs. Kronos Fine-tuned  (context=120, forecast=6, seed=42, test: 2021–)")
    print("="*65)
    print(df.to_string())
    print()
    if len(rows) == 2:
        base, ft = rows
        for m in ['MAE', 'IC', 'RankIC']:
            delta = ft[m] - base[m]
            sign  = '+' if delta >= 0 else ''
            better = '↑' if (m == 'MAE' and delta < 0) or (m != 'MAE' and delta > 0) else '↓'
            print(f"  {m:8s}  delta = {sign}{delta:.4f}  {better}")
